# CNN Image Classification Notebook

Notebook utama CNN untuk Kaggle dan lokal. Semua kebutuhan Kaggle yang dulu ada di launcher sekarang ada di notebook ini: discovery dataset Intel, prepare train/val/test, mode demo/full, output directory, training shared/non-shared CNN, evaluasi Keras vs scratch, visualisasi, dan bonus sanity check.

## 0. Setup

In [1]:
import csv
import json
import os
import random
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    if (current / "src" / "notebook").exists():
        return current
    for child in current.iterdir() if current.is_dir() else []:
        if child.is_dir() and (child / "src" / "notebook").exists():
            return child
    for parent in current.parents:
        if (parent / "src" / "notebook").exists():
            return parent
        for child in parent.iterdir() if parent.is_dir() else []:
            if child.is_dir() and (child / "src" / "notebook").exists():
                return child

    input_root = Path("/kaggle/input")
    code_slug = os.getenv("CODE_DATASET_SLUG", "").strip()
    search_roots = []
    if code_slug and (input_root / code_slug).exists():
        search_roots.append(input_root / code_slug)
    if input_root.exists():
        search_roots.append(input_root)

    for root in search_roots:
        candidates = [root, *root.glob("**/*")]
        for candidate in candidates:
            if candidate.is_dir() and (candidate / "src" / "notebook").exists():
                working_copy = Path("/kaggle/working/SpreiElsa")
                if Path("/kaggle/working").exists():
                    if working_copy.exists():
                        shutil.rmtree(working_copy)
                    shutil.copytree(candidate, working_copy)
                    return working_copy
                return candidate

    raise FileNotFoundError(
        "Repo root tidak ditemukan. Jalankan notebook dari root repo atau upload repo sebagai Kaggle Dataset "
        "dan, jika perlu, isi environment CODE_DATASET_SLUG."
    )


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

CNN_SRC = REPO_ROOT / "src" / "cnn"
UTILS_SRC = REPO_ROOT / "src" / "utils"
for path in (str(REPO_ROOT), str(CNN_SRC), str(UTILS_SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Repo root:", REPO_ROOT)
print("TensorFlow:", tf.__version__)

2026-05-14 06:08:48.170036: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778738928.325087     158 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778738928.370971     158 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778738928.725558     158 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778738928.725588     158 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778738928.725591     158 computation_placer.cc:177] computation placer alr

Repo root: /kaggle/working/SpreiElsa
TensorFlow: 2.19.0


In [2]:
from src.cnn.experiments import CLASS_NAMES, all_shared_experiments, get_experiment
from src.cnn.train_cnn import train_one
from src.cnn import compare_shared_non_shared, evaluate_cnn, plotting, scratch_compare, visualize_features
from src.dataset.prepare_intel_dataset import ensure_prepared_dataset, prepare_intel_dataset
from src.utils.image_utils import list_image_paths, load_images

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()


def env_flag(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


RUN_MODE = os.getenv("RUN_MODE", "full" if IS_KAGGLE else "local").strip().lower()
if RUN_MODE == "smoke":
    RUN_MODE = "demo"
if RUN_MODE not in {"local", "demo", "full"}:
    raise ValueError("RUN_MODE harus 'local', 'demo', atau 'full'")

DEMO_MODE = RUN_MODE == "demo"
DEMO_SYNTHETIC_DATASET = env_flag("DEMO_SYNTHETIC_DATASET", DEMO_MODE)

RUN_PREPARE = env_flag("RUN_PREPARE", IS_KAGGLE or DEMO_MODE)
RUN_TRAIN_SINGLE = env_flag("RUN_TRAIN_SINGLE", DEMO_MODE)
RUN_TRAIN_ALL = env_flag("RUN_TRAIN_ALL", IS_KAGGLE and RUN_MODE == "full")
RUN_NON_SHARED = env_flag("RUN_NON_SHARED", IS_KAGGLE and RUN_MODE == "full")
RUN_EVALUATE = env_flag("RUN_EVALUATE", IS_KAGGLE or DEMO_MODE)
RUN_VISUALIZE = env_flag("RUN_VISUALIZE", IS_KAGGLE or DEMO_MODE)

CNN_IMAGE_SIZE = int(os.getenv("CNN_IMAGE_SIZE", "32" if DEMO_MODE else "64"))
CNN_EXPERIMENT_ID = os.getenv("CNN_EXPERIMENT_ID", "d1_f16_k3_max")
CNN_NON_SHARED = env_flag("CNN_NON_SHARED", False)
CNN_EPOCHS = int(os.getenv("CNN_EPOCHS", "1" if DEMO_MODE else "10"))
CNN_BATCH_SIZE = int(os.getenv("CNN_BATCH_SIZE", "8" if DEMO_MODE else "32" if IS_KAGGLE else "16"))
SCRATCH_MAX_SAMPLES = int(os.getenv("SCRATCH_MAX_SAMPLES", "12" if DEMO_MODE else "30"))

DEMO_INTEL_TRAIN_PER_CLASS = int(os.getenv("DEMO_INTEL_TRAIN_PER_CLASS", "4"))
DEMO_INTEL_VAL_PER_CLASS = int(os.getenv("DEMO_INTEL_VAL_PER_CLASS", "2"))
DEMO_INTEL_TEST_PER_CLASS = int(os.getenv("DEMO_INTEL_TEST_PER_CLASS", "2"))

# Alias lama supaya cell lama tetap mudah dibaca.
cnn_image_size = CNN_IMAGE_SIZE
cnn_experiment_id = CNN_EXPERIMENT_ID
cnn_non_shared = CNN_NON_SHARED
cnn_epochs = CNN_EPOCHS
cnn_batch_size = CNN_BATCH_SIZE
run_prepare = RUN_PREPARE
run_train_single = RUN_TRAIN_SINGLE
run_train_all = RUN_TRAIN_ALL
run_non_shared = RUN_NON_SHARED

{
    "RUN_MODE": RUN_MODE,
    "IS_KAGGLE": IS_KAGGLE,
    "DEMO_SYNTHETIC_DATASET": DEMO_SYNTHETIC_DATASET,
    "RUN_PREPARE": RUN_PREPARE,
    "RUN_TRAIN_SINGLE": RUN_TRAIN_SINGLE,
    "RUN_TRAIN_ALL": RUN_TRAIN_ALL,
    "RUN_NON_SHARED": RUN_NON_SHARED,
    "RUN_EVALUATE": RUN_EVALUATE,
    "CNN_EPOCHS": CNN_EPOCHS,
    "CNN_BATCH_SIZE": CNN_BATCH_SIZE,
    "CNN_IMAGE_SIZE": CNN_IMAGE_SIZE,
}

{'RUN_MODE': 'full',
 'IS_KAGGLE': True,
 'DEMO_SYNTHETIC_DATASET': False,
 'RUN_PREPARE': True,
 'RUN_TRAIN_SINGLE': False,
 'RUN_TRAIN_ALL': True,
 'RUN_NON_SHARED': True,
 'RUN_EVALUATE': True,
 'CNN_EPOCHS': 10,
 'CNN_BATCH_SIZE': 32,
 'CNN_IMAGE_SIZE': 64}

## 0a. Path Resolution

In [4]:
INTEL_DATASET_SLUG = os.getenv("INTEL_DATASET_SLUG", "").strip()
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else REPO_ROOT / "outputs"
DEFAULT_WORKING_DIR = (
    WORK_ROOT / "outputs_demo" / "cnn" if IS_KAGGLE and DEMO_MODE else
    WORK_ROOT / "outputs" / "cnn" if IS_KAGGLE else
    WORK_ROOT / ("cnn_demo" if DEMO_MODE else "cnn")
)
WORKING_DIR = Path(os.getenv("WORKING_DIR", os.getenv("CNN_OUTPUT_DIR", DEFAULT_WORKING_DIR)))
PREPARED_ROOT = Path(
    os.getenv(
        "INTEL_PREPARED_ROOT",
        str(WORK_ROOT / "intel_prepared_demo") if DEMO_MODE else str(WORK_ROOT / "intel_prepared"),
    )
)
PLOTS_DIR = WORKING_DIR / "plots"
EVAL_DIR = WORKING_DIR / "eval"
VISUALS_DIR = WORKING_DIR / "visuals"

for p in (WORKING_DIR, PREPARED_ROOT, PLOTS_DIR, EVAL_DIR, VISUALS_DIR):
    p.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def optional_path(value):
    if value in (None, ""):
        return None
    return Path(value)


def path_from_slug(slug: str):
    if not IS_KAGGLE or not slug:
        return None
    path = INPUT_ROOT / slug
    return path if path.exists() else None


def find_dir(name_options, preferred_slug=""):
    roots = []
    explicit = path_from_slug(preferred_slug)
    if explicit:
        roots.append(explicit)
    if IS_KAGGLE and INPUT_ROOT.exists():
        roots.append(INPUT_ROOT)
    for root in roots:
        for name in name_options:
            matches = sorted(path for path in root.glob(f"**/{name}") if path.is_dir())
            if matches:
                return matches[0]
    return None


def has_prepared_layout(root: Path) -> bool:
    return root is not None and all((root / split).is_dir() for split in ("train", "val", "test"))


def has_intel_layout(root: Path) -> bool:
    return root is not None and all((root / split).is_dir() or (root / split / split).is_dir() for split in ("seg_train", "seg_test"))


def discover_intel_root() -> Path:
    env_root = optional_path(os.getenv("INTEL_DATA_ROOT"))
    if env_root:
        return env_root
    if not IS_KAGGLE:
        return REPO_ROOT / "src" / "dataset"

    candidates = [
        path_from_slug(INTEL_DATASET_SLUG),
        INPUT_ROOT / "intel-image-classification" if INPUT_ROOT.exists() else None,
        find_dir(["intel-image-classification"], INTEL_DATASET_SLUG),
    ]
    if INPUT_ROOT.exists():
        candidates.extend(path for path in INPUT_ROOT.iterdir() if path.is_dir())

    for candidate in candidates:
        if candidate and candidate.exists() and (has_prepared_layout(candidate) or has_intel_layout(candidate)):
            return candidate
    return INPUT_ROOT / "intel-image-classification"


INTEL_SOURCE_ROOT = discover_intel_root()
cnn_output_dir = WORKING_DIR
prepared_root = PREPARED_ROOT
intel_source_root = INTEL_SOURCE_ROOT


def create_synthetic_demo_dataset() -> Path:
    from PIL import Image

    rng = np.random.default_rng(SEED)
    demo_root = WORKING_DIR / "synthetic_intel"
    if has_prepared_layout(demo_root):
        return demo_root

    counts = {
        "train": DEMO_INTEL_TRAIN_PER_CLASS,
        "val": DEMO_INTEL_VAL_PER_CLASS,
        "test": DEMO_INTEL_TEST_PER_CLASS,
    }
    colors = np.array(
        [
            [180, 70, 70],
            [60, 150, 80],
            [90, 130, 210],
            [180, 150, 60],
            [60, 170, 190],
            [150, 90, 180],
        ],
        dtype=np.uint8,
    )
    for split, count in counts.items():
        for class_index, class_name in enumerate(CLASS_NAMES):
            class_dir = demo_root / split / class_name
            class_dir.mkdir(parents=True, exist_ok=True)
            for i in range(count):
                base = np.zeros((CNN_IMAGE_SIZE, CNN_IMAGE_SIZE, 3), dtype=np.uint8)
                base[:] = colors[class_index]
                noise = rng.integers(0, 35, size=base.shape, dtype=np.uint8)
                image = np.clip(base.astype(np.int16) + noise.astype(np.int16), 0, 255).astype(np.uint8)
                image[i % CNN_IMAGE_SIZE, :, :] = 255 - colors[class_index]
                Image.fromarray(image).save(class_dir / f"{class_name}_{i:03d}.jpg")
    print("Synthetic demo dataset:", demo_root)
    return demo_root


for key, value in {
    "INTEL_DATA_ROOT": INTEL_SOURCE_ROOT,
    "INTEL_PREPARED_ROOT": PREPARED_ROOT,
    "CNN_OUTPUT_DIR": WORKING_DIR,
}.items():
    if value is not None:
        os.environ[key] = str(value)

{
    "RUN_MODE": RUN_MODE,
    "INTEL_SOURCE_ROOT": str(INTEL_SOURCE_ROOT),
    "PREPARED_ROOT": str(PREPARED_ROOT),
    "WORKING_DIR": str(WORKING_DIR),
}

{'RUN_MODE': 'full',
 'INTEL_SOURCE_ROOT': '/kaggle/input/datasets/puneet6060/intel-image-classification',
 'PREPARED_ROOT': '/kaggle/working/intel_prepared',
 'WORKING_DIR': '/kaggle/working/outputs/cnn'}

In [5]:
def read_json(path: Path, default=None):
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)


def has_dataset(root: Path) -> bool:
    if not has_prepared_layout(root):
        print("Dataset belum tersedia dalam layout train/val/test:", root)
        return False
    return True

## 1. Dataset Preparation

In [6]:
if DEMO_MODE and DEMO_SYNTHETIC_DATASET:
    cnn_data_root = create_synthetic_demo_dataset()
elif has_prepared_layout(PREPARED_ROOT):
    cnn_data_root = PREPARED_ROOT
elif has_prepared_layout(INTEL_SOURCE_ROOT):
    cnn_data_root = INTEL_SOURCE_ROOT
elif DEMO_MODE and INTEL_SOURCE_ROOT.exists() and has_intel_layout(INTEL_SOURCE_ROOT):
    prepare_intel_dataset(
        source_root=INTEL_SOURCE_ROOT,
        output_root=PREPARED_ROOT,
        val_fraction=0.15,
        overwrite=False,
        max_train_per_class=DEMO_INTEL_TRAIN_PER_CLASS,
        max_val_per_class=DEMO_INTEL_VAL_PER_CLASS,
        max_test_per_class=DEMO_INTEL_TEST_PER_CLASS,
    )
    cnn_data_root = PREPARED_ROOT
elif RUN_PREPARE and INTEL_SOURCE_ROOT.exists():
    cnn_data_root = ensure_prepared_dataset(
        data_root=INTEL_SOURCE_ROOT,
        prepared_root=PREPARED_ROOT,
        val_fraction=0.15,
        overwrite=False,
    )
else:
    cnn_data_root = INTEL_SOURCE_ROOT
    print(
        "Dataset CNN belum dalam layout train/val/test. "
        "Set RUN_PREPARE=True atau INTEL_DATA_ROOT ke folder Intel yang berisi seg_train dan seg_test."
    )

print("cnn_data_root:", cnn_data_root)
cnn_data_root

cnn_data_root: /kaggle/working/intel_prepared


PosixPath('/kaggle/working/intel_prepared')

## 2. Sanity Check Loader

In [7]:
if has_dataset(cnn_data_root):
    paths, labels, classes = list_image_paths(cnn_data_root / "train", class_names=CLASS_NAMES)
    sample_paths = paths[:16]
    sample_images = load_images(sample_paths, image_size=(cnn_image_size, cnn_image_size), normalize=True)
    (sample_images.shape, labels[:16].shape, classes)
else:
    sample_images = np.empty((0, cnn_image_size, cnn_image_size, 3), dtype=np.float32)

## 3. Experiment Grid

In [8]:
configs = all_shared_experiments(image_size=CNN_IMAGE_SIZE)
config_rows = [cfg.to_dict() for cfg in configs]
config_rows[:4]

[{'experiment_id': 'd1_f16_k3_max',
  'conv_depth': 1,
  'filters_base': 16,
  'kernel_size': 3,
  'pooling': 'max',
  'image_size': 64,
  'num_classes': 6,
  'dense_units': 64,
  'learning_rate': 0.001,
  'filters': [16]},
 {'experiment_id': 'd1_f16_k3_avg',
  'conv_depth': 1,
  'filters_base': 16,
  'kernel_size': 3,
  'pooling': 'avg',
  'image_size': 64,
  'num_classes': 6,
  'dense_units': 64,
  'learning_rate': 0.001,
  'filters': [16]},
 {'experiment_id': 'd1_f16_k5_max',
  'conv_depth': 1,
  'filters_base': 16,
  'kernel_size': 5,
  'pooling': 'max',
  'image_size': 64,
  'num_classes': 6,
  'dense_units': 64,
  'learning_rate': 0.001,
  'filters': [16]},
 {'experiment_id': 'd1_f16_k5_avg',
  'conv_depth': 1,
  'filters_base': 16,
  'kernel_size': 5,
  'pooling': 'avg',
  'image_size': 64,
  'num_classes': 6,
  'dense_units': 64,
  'learning_rate': 0.001,
  'filters': [16]}]

## 4. Train Single Experiment

In [9]:
if RUN_TRAIN_SINGLE and has_dataset(cnn_data_root):
    config = get_experiment(CNN_EXPERIMENT_ID, image_size=CNN_IMAGE_SIZE)
    exp_dir = train_one(
        config=config,
        data_root=cnn_data_root,
        output_dir=WORKING_DIR,
        epochs=CNN_EPOCHS,
        batch_size=CNN_BATCH_SIZE,
        non_shared=CNN_NON_SHARED,
    )
    exp_dir
else:
    print("RUN_TRAIN_SINGLE=False atau dataset belum siap.")

RUN_TRAIN_SINGLE=False atau dataset belum siap.


## 5. Train All Shared Experiments

In [10]:
if RUN_TRAIN_ALL and has_dataset(cnn_data_root):
    for config in all_shared_experiments(image_size=CNN_IMAGE_SIZE):
        train_one(
            config=config,
            data_root=cnn_data_root,
            output_dir=WORKING_DIR,
            epochs=CNN_EPOCHS,
            batch_size=CNN_BATCH_SIZE,
            non_shared=False,
        )
    print("Selesai run_all shared experiments")
else:
    print("RUN_TRAIN_ALL=False atau dataset belum tersedia.")

Found 11932 files belonging to 6 classes.


I0000 00:00:1778739110.065372     158 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778739110.071335     158 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 2102 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.
Epoch 1/10


I0000 00:00:1778739113.599670     219 service.cc:152] XLA service 0x7aaf500034e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778739113.599718     219 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778739113.599723     219 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778739113.865661     219 cuda_dnn.cc:529] Loaded cuDNN version 91002


 34/373 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2935 - loss: 1.6355

I0000 00:00:1778739115.656145     219 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


373/373 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.5387 - loss: 1.1745 - val_accuracy: 0.6989 - val_loss: 0.8436
Epoch 2/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7304 - loss: 0.7389 - val_accuracy: 0.7226 - val_loss: 0.7619
Epoch 3/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7921 - loss: 0.5677 - val_accuracy: 0.7160 - val_loss: 0.8185
Epoch 4/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8370 - loss: 0.4572 - val_accuracy: 0.7260 - val_loss: 0.8082
Epoch 5/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8818 - loss: 0.3470 - val_accuracy: 0.7131 - val_loss: 0.9622
Epoch 6/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9134 - loss: 0.2589 - val_accuracy: 0.7288 - val_loss: 0.9682
Epoch 7/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9441 - loss: 0.1905 - val_accuracy: 0.7136 - val_loss: 1.0819
Epoch 8/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9592 - loss: 0.1457 - val_accuracy: 0.7188 - val

## 6. Summarize Experiments + Pick Best

In [11]:
summary_rows = plotting.summarize_experiments(WORKING_DIR, WORKING_DIR / "summary.csv")
best_row = summary_rows[0] if summary_rows else None
best_row

{'experiment': 'shared_d2_f32_k3_max',
 'conv_depth': 2,
 'filters_base': 32,
 'kernel_size': 3,
 'pooling': 'max',
 'non_shared': False,
 'macro_f1': 0.7891029230446035,
 'test_accuracy': 0.7883333563804626,
 'param_count': 822662}

## 7. Evaluate Best + Scratch Compare

In [12]:
eval_metrics = None
scratch_metrics = None
if RUN_EVALUATE and best_row and has_dataset(cnn_data_root):
    model_dir = WORKING_DIR / best_row["experiment"]
    model_path = model_dir / "model.keras"
    if model_path.exists():
        eval_metrics = evaluate_cnn.evaluate_model(
            model_path=model_path,
            data_root=cnn_data_root,
            output_dir=EVAL_DIR,
            image_size=CNN_IMAGE_SIZE,
            batch_size=CNN_BATCH_SIZE,
        )
        scratch_metrics = scratch_compare.compare_keras_scratch(
            model_path=model_path,
            split_root=cnn_data_root / "test",
            output_path=WORKING_DIR / "scratch_compare.json",
            image_size=CNN_IMAGE_SIZE,
            max_samples=SCRATCH_MAX_SAMPLES,
        )
    else:
        print("Model terbaik belum ada di", model_dir)
else:
    print("RUN_EVALUATE=False, belum ada summary, atau dataset test belum siap.")

eval_metrics, scratch_metrics

Found 3000 files belonging to 6 classes.


({'macro_f1': 0.7891029230446035,
  'param_count': 822662,
  'confusion_matrix': [[313, 11, 16, 13, 25, 59],
   [7, 442, 3, 4, 5, 13],
   [7, 4, 424, 70, 43, 5],
   [7, 1, 74, 408, 33, 2],
   [13, 2, 50, 39, 394, 12],
   [68, 23, 12, 3, 11, 384]],
  'class_names': ['buildings',
   'forest',
   'glacier',
   'mountain',
   'sea',
   'street']},
 {'samples': 30,
  'keras_macro_f1': 0.17358490566037738,
  'scratch_macro_f1': 0.17358490566037738,
  'max_abs_probability_diff': 5.364418029785156e-07,
  'prediction_match_rate': 1.0})

## 8. Shared vs Non-Shared Comparison

In [13]:
def strip_experiment_prefix(name: str) -> str:
    for prefix in ("shared_", "non_shared_"):
        if name.startswith(prefix):
            return name[len(prefix):]
    return name


comparison = None
if best_row:
    exp_id = strip_experiment_prefix(best_row["experiment"])
    shared_dir = WORKING_DIR / f"shared_{exp_id}"
    non_shared_dir = WORKING_DIR / f"non_shared_{exp_id}"
    if RUN_NON_SHARED and has_dataset(cnn_data_root):
        config = get_experiment(exp_id, image_size=CNN_IMAGE_SIZE)
        train_one(
            config=config,
            data_root=cnn_data_root,
            output_dir=WORKING_DIR,
            epochs=CNN_EPOCHS,
            batch_size=CNN_BATCH_SIZE,
            non_shared=True,
        )
    if shared_dir.exists() and non_shared_dir.exists():
        comparison = compare_shared_non_shared.compare(
            shared_dir=shared_dir,
            non_shared_dir=non_shared_dir,
            output_path=WORKING_DIR / "shared_vs_non_shared.csv",
        )
    else:
        print("Shared/non-shared artifacts belum lengkap.")
else:
    print("Best row belum tersedia.")

comparison

Found 11932 files belonging to 6 classes.
Found 2102 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.
Epoch 1/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.3947 - loss: 1.4789 - val_accuracy: 0.5314 - val_loss: 1.2056
Epoch 2/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.5480 - loss: 1.1655 - val_accuracy: 0.6004 - val_loss: 1.0991
Epoch 3/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.5926 - loss: 1.0520 - val_accuracy: 0.6475 - val_loss: 0.9618
Epoch 4/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.6302 - loss: 0.9547 - val_accuracy: 0.6232 - val_loss: 0.9609
Epoch 5/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.6575 - loss: 0.8858 - val_accuracy: 0.6594 - val_loss: 0.9308
Epoch 6/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.6926 - loss: 0.8196 - val_accuracy: 0.6808 - val_loss: 0.8775
Epoch 7/10
373/373 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.7021 - loss: 0.7934 - val_accuracy

{'rows': [{'model_type': 'shared',
   'experiment': 'shared_d2_f32_k3_max',
   'macro_f1': 0.7891029230446035,
   'test_accuracy': 0.7883333563804626,
   'param_count': 822662},
  {'model_type': 'non_shared',
   'experiment': 'non_shared_d2_f32_k3_max',
   'macro_f1': 0.6443638286579291,
   'test_accuracy': 0.656333327293396,
   'param_count': 19802630}]}

## 9. Loss Curves + Summary Table

In [14]:
if summary_rows:
    top5 = summary_rows[:5]
    top5
else:
    print("Belum ada summary. Jalankan training/summarize dulu.")

loss_examples = []
if summary_rows:
    for row in summary_rows[:3]:
        loss_path = WORKING_DIR / row["experiment"] / "loss.png"
        if loss_path.exists():
            loss_examples.append(loss_path)
loss_examples

[PosixPath('/kaggle/working/outputs/cnn/shared_d2_f32_k3_max/loss.png'),
 PosixPath('/kaggle/working/outputs/cnn/shared_d2_f32_k3_avg/loss.png'),
 PosixPath('/kaggle/working/outputs/cnn/shared_d2_f16_k3_max/loss.png')]

## 10. Bonus: Feature Maps + Grad-CAM

In [15]:
viz_image_path = None
viz_layer_name = None  # isi nama layer conv jika mau spesifik

if has_prepared_layout(cnn_data_root):
    test_paths, _, _ = list_image_paths(cnn_data_root / "test", class_names=CLASS_NAMES)
    viz_image_path = test_paths[0] if test_paths else None

if RUN_VISUALIZE and best_row:
    model_path = WORKING_DIR / best_row["experiment"] / "model.keras"
    if model_path.exists() and viz_image_path is not None and viz_image_path.exists():
        visualize_features.visualize(
            model_path=model_path,
            image_path=viz_image_path,
            output_dir=VISUALS_DIR,
            image_size=CNN_IMAGE_SIZE,
            layer_name=viz_layer_name,
            class_index=None,
            max_channels=32,
            alpha=0.4,
        )
        print("Saved visuals to", VISUALS_DIR)
    else:
        print("Model atau image belum ada.")
else:
    print("RUN_VISUALIZE=False atau best row belum tersedia.")

Saved visuals to /kaggle/working/outputs/cnn/visuals


## 11. Bonus: Batch Inference Sanity (Scratch)

In [16]:
from src.cnn.cnn_scratch.layers import Conv2D, Dense, Flatten, LocallyConnected2D, MaxPooling2D
from src.cnn.cnn_scratch.model import SequentialScratchModel

def check_shared_batch() -> None:
    rng = np.random.default_rng(0)
    x = rng.normal(size=(4, 8, 8, 3)).astype(np.float32)
    kernel = rng.normal(size=(3, 3, 3, 4)).astype(np.float32)
    bias = np.zeros((4,), dtype=np.float32)

    conv = Conv2D(kernel=kernel, bias=bias, activation="relu")
    pool = MaxPooling2D(pool_size=(2, 2))
    flat = Flatten()
    dense = Dense(rng.normal(size=(3 * 3 * 4, 5)).astype(np.float32), np.zeros((5,), dtype=np.float32))

    model = SequentialScratchModel([conv, pool, flat, dense])
    out = model.predict(x)
    assert out.shape == (4, 5), out.shape
    print("Shared batch output shape:", out.shape)

def check_non_shared_batch() -> None:
    rng = np.random.default_rng(1)
    x = rng.normal(size=(3, 8, 8, 3)).astype(np.float32)
    kh, kw, cin, cout = 3, 3, 3, 2
    out_h = (8 - kh) + 1
    out_w = (8 - kw) + 1
    positions = out_h * out_w
    kernel = rng.normal(size=(positions, kh * kw * cin, cout)).astype(np.float32)
    bias = np.zeros((positions, cout), dtype=np.float32)

    local = LocallyConnected2D(kernel=kernel, bias=bias, kernel_size=(kh, kw), activation="relu")
    out = local.forward(x)
    assert out.shape == (3, out_h, out_w, cout), out.shape
    print("Non-shared batch output shape:", out.shape)

check_shared_batch()
check_non_shared_batch()

Shared batch output shape: (4, 5)
Non-shared batch output shape: (3, 6, 6, 2)


## 12. Bonus: Backward Propagation Sanity (Scratch)

In [17]:
from src.cnn.cnn_scratch.layers import Conv2D as ScratchConv2D, Dense as ScratchDense
from src.cnn.cnn_scratch.losses import softmax_cross_entropy_loss

def _relative_error(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    denom = np.maximum(eps, np.maximum(np.abs(a), np.abs(b)))
    return float(np.max(np.abs(a - b) / denom))

def _dense_loss(x: np.ndarray, kernel: np.ndarray, bias: np.ndarray) -> float:
    out = x @ kernel + bias
    return float(np.sum(out))

def check_dense_backward() -> None:
    rng = np.random.default_rng(0)
    x = rng.normal(size=(2, 4)).astype(np.float32)
    kernel = rng.normal(size=(4, 3)).astype(np.float32)
    bias = rng.normal(size=(3,)).astype(np.float32)

    layer = ScratchDense(kernel.copy(), bias.copy(), activation=None)
    out = layer.forward(x)
    upstream = np.ones_like(out)
    layer.backward(upstream)

    eps = 1e-4
    grad_num = np.zeros_like(kernel)
    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            orig = kernel[i, j]
            kernel[i, j] = orig + eps
            loss_pos = _dense_loss(x, kernel, bias)
            kernel[i, j] = orig - eps
            loss_neg = _dense_loss(x, kernel, bias)
            grad_num[i, j] = (loss_pos - loss_neg) / (2 * eps)
            kernel[i, j] = orig

    err = _relative_error(layer.grad_kernel, grad_num)
    print("Dense grad kernel rel err:", err)

def _conv_loss(x: np.ndarray, kernel: np.ndarray) -> float:
    layer = ScratchConv2D(kernel=kernel, bias=None, strides=(1, 1), padding="valid", activation=None)
    out = layer.forward(x)
    return float(np.sum(out))

def check_conv2d_backward() -> None:
    rng = np.random.default_rng(1)
    x = rng.normal(size=(1, 4, 4, 1)).astype(np.float32)
    kernel = rng.normal(size=(3, 3, 1, 1)).astype(np.float32)

    layer = ScratchConv2D(kernel=kernel.copy(), bias=None, strides=(1, 1), padding="valid", activation=None)
    out = layer.forward(x)
    upstream = np.ones_like(out)
    layer.backward(upstream)

    eps = 1e-4
    grad_num = np.zeros_like(kernel)
    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            for c in range(kernel.shape[2]):
                for k in range(kernel.shape[3]):
                    orig = kernel[i, j, c, k]
                    kernel[i, j, c, k] = orig + eps
                    loss_pos = _conv_loss(x, kernel)
                    kernel[i, j, c, k] = orig - eps
                    loss_neg = _conv_loss(x, kernel)
                    grad_num[i, j, c, k] = (loss_pos - loss_neg) / (2 * eps)
                    kernel[i, j, c, k] = orig

    err = _relative_error(layer.grad_kernel, grad_num)
    print("Conv2D grad kernel rel err:", err)

check_dense_backward()
check_conv2d_backward()
logits = np.array([[1.0, -1.0, 0.5]], dtype=np.float32)
labels = np.array([2], dtype=np.int64)
loss, grad = softmax_cross_entropy_loss(logits, labels)
print("Softmax CE loss:", loss, "grad shape:", grad.shape)

Dense grad kernel rel err: 0.013040013611316681
Conv2D grad kernel rel err: 0.004356682300567627
Softmax CE loss: 1.0549569129943848 grad shape: (1, 3)


## 13. Bonus: Forward Sanity (Scratch)

In [18]:
from src.cnn.cnn_scratch.activations import softmax
from src.cnn.cnn_scratch.layers import Conv2D as ScratchConv2D2, Dense as ScratchDense2, Flatten as ScratchFlatten, MaxPooling2D as ScratchMaxPool

x = np.arange(1 * 4 * 4 * 1, dtype=np.float32).reshape((1, 4, 4, 1))
kernel = np.ones((3, 3, 1, 2), dtype=np.float32)
bias = np.array([0.0, 1.0], dtype=np.float32)
conv = ScratchConv2D2(kernel, bias=bias, activation="relu")
conv_out = conv.forward(x)
assert conv_out.shape == (1, 2, 2, 2), conv_out.shape

pool = ScratchMaxPool(pool_size=(2, 2))
pool_out = pool.forward(x)
assert pool_out.tolist() == [[[[5.0], [7.0]], [[13.0], [15.0]]]], pool_out

flat = ScratchFlatten().forward(x)
assert flat.shape == (1, 16)
assert flat[0, 0] == 0 and flat[0, -1] == 15

dense = ScratchDense2(np.ones((16, 3), dtype=np.float32), np.zeros((3,), dtype=np.float32), activation="softmax")
probs = dense.forward(flat)
assert probs.shape == (1, 3)
assert np.allclose(np.sum(probs, axis=1), 1.0)
assert np.allclose(np.sum(softmax(np.array([[1.0, 2.0, 3.0]])), axis=1), 1.0)
print("scratch sanity checks passed")

scratch sanity checks passed


## 14. Export Summary Artifacts

In [19]:
summary_payload = {
    "best_experiment": best_row,
    "summary_rows": summary_rows,
    "eval_metrics": eval_metrics,
    "scratch_metrics": scratch_metrics,
    "shared_vs_non_shared": comparison,
    "loss_examples": [str(path) for path in loss_examples],
    "data_root": str(cnn_data_root),
    "working_dir": str(WORKING_DIR),
}
write_json(WORKING_DIR / "notebook_summary.json", summary_payload)
WORKING_DIR / "notebook_summary.json"

PosixPath('/kaggle/working/outputs/cnn/notebook_summary.json')

## 15. Final Checklist

- Pastikan `RUN_MODE="demo"` selesai dulu untuk smoke test.
- Untuk eksperimen final di Kaggle, pakai `RUN_MODE="full"` dan pastikan dataset Intel tersedia di `/kaggle/input` atau isi `INTEL_DATASET_SLUG` / `INTEL_DATA_ROOT`.
- Hasil utama tersimpan di `WORKING_DIR`, termasuk `summary.csv`, `notebook_summary.json`, model, metrics, dan visualisasi.